In [1]:
!pip install transformers datasets evaluate accelerate scikit-learn gradio torch

In [ ]:
!pip install evaluate

In [ ]:
import numpy as np
import pandas as pd
import torch
import evaluate
import gradio as gr
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# ==========================================
# 1. DATASET LOADING & SUBSET SELECTION
# ==========================================

# FIX: Use full namespace path 'fancyzhx/ag_news' instead of 'ag_news'
dataset = load_dataset("fancyzhx/ag_news")

# Shuffle and slice a smaller subset to optimize training time and compute resources
small_train_data = dataset["train"].shuffle(seed=42).select(range(5000))
small_test_data = dataset["test"].shuffle(seed=42).select(range(1000))

# ==========================================
# 2. TOKENIZATION & PREPROCESSING
# ==========================================

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=64)

tokenized_train = small_train_data.map(tokenize_function, batched=True)
tokenized_test = small_test_data.map(tokenize_function, batched=True)

# ==========================================
# 3. MODEL CONFIGURATION & TRAINING ARGS
# ==========================================

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=4)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=100,
)

# ==========================================
# 4. EVALUATION METRICS DEFINITION
# ==========================================

metric_acc = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = metric_acc.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = metric_f1.compute(predictions=predictions, references=labels, average="weighted")["f1"]

    return {"accuracy": acc, "f1": f1}

# ==========================================
# 5. MODEL TRAINING & SAVING FLOW
# ==========================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

print("Starting training process...")
trainer.train()

print("Saving fine-tuned model artifacts...")
model.save_pretrained("./my_bert_model")
tokenizer.save_pretrained("./my_bert_model")
print("Model artifacts successfully saved!")

# ==========================================
# 6. APP INFERENCE LOGIC (GRADIO WEB DEPLOYMENT)
# ==========================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_path = "./my_bert_model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)

labels = ["World", "Sports", "Business", "Sci/Tech"]

def classify_news(headline):
    inputs = tokenizer(headline, return_tensors="pt", truncation=True, padding=True, max_length=64)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probabilities = torch.nn.functional.softmax(logits, dim=-1)[0]

    return {labels[i]: float(probabilities[i]) for i in range(4)}

demo = gr.Interface(
    fn=classify_news,
    inputs=gr.Textbox(lines=2, placeholder="Type an English news headline here...", label="News Headline"),
    outputs=gr.Label(num_top_classes=4, label="Predicted Category"),
    title="📰 News Topic Classifier (BERT)",
    description="Fine-tuned BERT model classifying news headlines into World, Sports, Business, or Sci/Tech classes."
)

if __name__ == "__main__":
    demo.launch(share=True)